# Phase 2 — QLoRA fine-tuning

**Before running: switch this Colab runtime to a GPU.** Runtime menu -> Change runtime type -> T4 GPU (free tier) or A100 (Pro). Everything in `01_explore_data.ipynb` ran fine on CPU; this notebook will not.

This trains a LoRA adapter on top of 4-bit quantized Qwen3-4B for one task at a time (extraction or summarization), tracked in MLflow. Assumes you've already run `01_explore_data.ipynb` at least once, so `data/processed/finetune_*.jsonl` already exist in this Drive folder — if not, run that notebook first.

## 1. Mount Drive and pull the repo

Same Drive-persisted setup as the exploration notebook, so checkpoints survive a runtime disconnect.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

from google.colab import userdata

GH_TOKEN = userdata.get("GH_TOKEN")
REPO_DIR = "/content/drive/MyDrive/Clinical_Report_Assistant"
REPO_URL = f"https://{GH_TOKEN}@github.com/ozgurberat/clinical-report-assistant.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already exists in Drive — pulling latest instead of re-cloning.")
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} "{REPO_DIR}"
    %cd {REPO_DIR}

## 2. Install the fine-tuning stack

Heavier than the exploration notebook's deps — this pulls in PyTorch's quantization/PEFT/RL training stack. Colab already has `torch` preinstalled; the rest need adding.

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes mlflow datasets pyyaml

## 3. Sanity-check the GPU is actually attached

If this errors or shows no GPU, go back and fix the runtime type before continuing — training will fail (or silently run on CPU at unusable speed) otherwise.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — check Runtime > Change runtime type"
print(torch.cuda.get_device_name(0))

## 4. Train the extraction task

This is the first real training run — first time this exact script executes anywhere, so treat the first attempt as a debugging pass rather than expecting it to run clean start to finish. Common first-run failures: a TRL/transformers API name mismatch (library versions move fast), an out-of-memory error (fixable by lowering `per_device_train_batch_size` or raising `gradient_accumulation_steps` in `configs/training.yaml`), or the response-template detection printing something unexpected. Whatever comes up, paste the error back and we'll fix it together.

In [ ]:
!python -m src.finetuning.train --task extraction

## 5. Train the summarization task

Same script, different `--task`. Run this once the extraction run above completes successfully.

In [ ]:
!python -m src.finetuning.train --task summarization

## 6. View MLflow results

MLflow logs runs locally to an `mlruns/` folder by default. Launch its UI inline in Colab:

In [ ]:
get_ipython().system_raw("mlflow ui --port 5000 &")

from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(5000)"))

Open the printed URL to see both runs side by side — loss curves, logged hyperparameters (LoRA rank, learning rate, epochs), and example counts. This is the comparison view we'll use later to justify hyperparameter choices in the README results table.

## 7. Qualitative check — read actual generated outputs

Loss and accuracy numbers are proxy metrics. Before calling either fine-tune "done," we need to actually read what the model generates on held-out test examples and judge whether it's genuinely good, not just numerically low-loss. This loads each saved adapter on top of the base model and generates real outputs for a few test-set examples, printed next to the ground truth for direct comparison.

**Result from the first run:** extraction was a perfect exact-match on all 3 test examples (genuine generalization, not memorization — these are held-out test rows). Summarization was 1/3 exact and 2/3 differently-worded-but-clinically-equivalent (e.g. "No acute process." vs "No acute pulmonary findings.") — the expected, healthy outcome for a real generation task, not a problem.

One artifact the first run surfaced: every output was prefixed with an empty `<think>\n\n</think>\n\n` block. Qwen3 has a built-in "thinking mode" that's on by default in its chat template; since our fine-tuning data never modeled an actual reasoning trace (we trained input -> answer directly), the model just emits an empty thinking block out of habit before the real content. Harmless to correctness, but wasted tokens and something any real consumer of this model would have to strip out — so we suppress it at generation time with `enable_thinking=False` below. Worth remembering this for Phase 4 serving too.

In [ ]:
import itertools
import json

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen3-4B"


def load_finetuned(task):
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto"
    )
    adapter_dir = f"outputs/{task}-Qwen3-4B/final_adapter"
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    return model, tokenizer


def generate_reply(model, tokenizer, messages, max_new_tokens=256):
    # Only system + user go in — the model generates what would be the assistant turn.
    # enable_thinking=False suppresses Qwen3's default reasoning-trace mode, which
    # our fine-tuning never modeled (see markdown note above).
    prompt = tokenizer.apply_chat_template(
        messages[:2],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
    # Belt-and-suspenders: strip any think block that slips through anyway.
    import re

    return re.sub(r"<think>.*?</think>\s*", "", text, flags=re.DOTALL).strip()


for task in ["extraction", "summarization"]:
    print(f"\n=== {task} ===")
    model, tokenizer = load_finetuned(task)
    with open(f"data/processed/finetune_{task}_test.jsonl") as f:
        examples = [json.loads(line) for line in itertools.islice(f, 3)]
    for ex in examples:
        prediction = generate_reply(model, tokenizer, ex["messages"])
        print(f"--- report_id {ex['report_id']} ---")
        print("GROUND TRUTH:", ex["messages"][2]["content"])
        print("PREDICTED   :", prediction)
        print()
    del model
    torch.cuda.empty_cache()

## Next

Read through the printed predictions vs. ground truth for both tasks, and confirm the think-tag prefix is now gone. If both still look reasonable, Phase 2's fine-tuning is functionally done, and the remaining work is turning this into real metrics for the results table (exact-match/F1 on the extraction JSON fields, ROUGE-L on summarization) rather than eyeballing a handful of examples. If anything looks clearly wrong (garbled JSON, hallucinated content, repeated text), bring back specific examples and we'll dig into why.